In [20]:
from gurobipy import Model, GRB, quicksum
import math 

# =============================================================================
# 1) DATA & SETS
# =============================================================================

W = ["W"]
Hubs = ["YUL", "LGR", "Puvirnituq", "Kuujjuaq"]

# The 14 Nunavik Communities
Communities = [
    "Akulivik", "Ivujivik", "Salluit",  # Served by Puvirnituq
    "Kuujjuaraapik", "Sanikiluaq", "Umiujaq", "Inukjuak", # Served by LGR
    "Quaqtaq", "Kangiqsualujjuaq", "Tasiujaq", "Aupaluk", "Kangirsuk", "Kangiqsujuaq", # Served by Kuujjuaq
    "Kuujjuaq", "Puvirnituq" # Hubs
]

# --- PRODUCT LIST (Fruits & Veggies Only) ---
kg_to_lbs = 2.20462
demand_per_person_kg = {
    "Oranges": 0.3075, "Apples": 1.095, "Bananas": 0.895, "Grapes": 0.125,
    "Carrots": 0.50, "Onions": 0.17375, "Cabbage": 0.13, "Turnips": 0.0875
}
Products = list(demand_per_person_kg.keys())

# --- PALLET PHYSICS (Lbs per Pallet) ---
pallet_weights = {
    "Oranges": 3600.0, "Apples": 3600.0, 
    "Bananas": 1920.0, "Grapes": 2400.0, 
    "Carrots": 4000.0, "Onions": 4000.0, 
    "Cabbage": 2400.0, "Turnips": 2500.0
}

# --- SPOILAGE RATES (Specific to Produce) ---
spoil_rate = {
    "Oranges": 0.002, "Apples": 0.002, "Bananas": 0.008, 
    "Grapes": 0.005, "Carrots": 0.001, "Onions": 0.001,
    "Cabbage": 0.002, "Turnips": 0.001
}

# --- POPULATION DATA (2024 Est) ---
population = {
    "Kuujjuaq": 2800, "Puvirnituq": 1800, "Inukjuak": 1800, "Salluit": 1500,
    "Kangiqsujuaq": 850, "Kuujjuaraapik": 800, "Akulivik": 700, "Kangirsuk": 600,
    "Kangiqsualujjuaq": 950, "Tasiujaq": 350, "Aupaluk": 250, "Umiujaq": 500,
    "Ivujivik": 450, "Quaqtaq": 450, "Sanikiluaq": 900
}

# --- CALCULATE TOTAL DEMAND (Lbs) ---
demand_input = {}
for c in Communities:
    pop = population.get(c, 500)
    for p, kg in demand_per_person_kg.items():
        demand_input[(c, p)] = pop * kg * kg_to_lbs

TOTAL_DEMAND = sum(demand_input.values())

# =============================================================================
# 2) NETWORK DEFINITION
# =============================================================================

# --- TRANSIT TIMES (Hours) ---
transit_time = {
    ("W", "YUL"): 0.33, ("W", "LGR"): 16.0,
    ("YUL", "Kuujjuaq"): 2.33, ("YUL", "Puvirnituq"): 2.50,
    ("Puvirnituq", "Akulivik"): 0.5, ("Puvirnituq", "Ivujivik"): 1.0, 
    ("Puvirnituq", "Salluit"): 1.0, ("Puvirnituq", "Puvirnituq"): 0.0,
    ("LGR", "Kuujjuaraapik"): 0.5, ("LGR", "Sanikiluaq"): 1.5,
    ("LGR", "Umiujaq"): 1.0, ("LGR", "Inukjuak"): 1.5,
    ("Kuujjuaq", "Quaqtaq"): 1.0, ("Kuujjuaq", "Kangiqsualujjuaq"): 0.75,
    ("Kuujjuaq", "Tasiujaq"): 0.5, ("Kuujjuaq", "Aupaluk"): 0.75,
    ("Kuujjuaq", "Kangirsuk"): 0.75, ("Kuujjuaq", "Kangiqsujuaq"): 1.0,
    ("Kuujjuaq", "Kuujjuaq"): 0.0
}

# --- CAPACITIES ---
TRUCK_MAX_WEIGHT = 44092.0
TRUCK_MAX_PALLETS = 26.0
hub_capacity = {"YUL": 350000.0, "LGR": 91000.0} 

# --- RISK & DELAYS ---
cancel_prob = {k: 0.05 for k in transit_time} 
risk_update = {
    ("Puvirnituq", "Ivujivik"): 0.37, ("Puvirnituq", "Salluit"): 0.15,
    ("LGR", "Sanikiluaq"): 0.15, ("LGR", "Ivujivik"): 0.37, ("YUL", "Puvirnituq"): 0.12
}
cancel_prob.update(risk_update)
cancel_prob[("W", "YUL")] = 0.0
cancel_prob[("W", "LGR")] = 0.0

full_prob = {"YUL": 0.15, "LGR": 0.25}
wait_time = {"YUL": 4.0, "LGR": 6.0}

# =============================================================================
# 3) PRE-COMPUTATION
# =============================================================================

eff_time = {}
for arc, t in transit_time.items():
    p = cancel_prob.get(arc, 0.0)
    t_risk = t * (1.0 + p) / (1.0 - p) if p < 0.99 else 1e6
    if arc == ("W", "YUL"): t_risk += full_prob["YUL"] * wait_time["YUL"]
    if arc == ("W", "LGR"): t_risk += full_prob["LGR"] * wait_time["LGR"]
    eff_time[arc] = t_risk

routes = []
hub_map = {c: "Puvirnituq" for c in ["Akulivik", "Ivujivik", "Salluit", "Puvirnituq"]}
hub_map.update({c: "LGR" for c in ["Kuujjuaraapik", "Sanikiluaq", "Umiujaq", "Inukjuak"]})
hub_map.update({c: "Kuujjuaq" for c in ["Quaqtaq", "Kangiqsualujjuaq", "Tasiujaq", "Aupaluk", "Kangirsuk", "Kangiqsujuaq", "Kuujjuaq"]})

for c in Communities:
    reg_hub = hub_map[c]
    for entry in ["YUL", "LGR"]:
        path_arcs = [("W", entry)]
        if entry != reg_hub:
            if (entry, reg_hub) in eff_time: path_arcs.append((entry, reg_hub))
            else: continue 
        if reg_hub != c:
            if (reg_hub, c) in eff_time: path_arcs.append((reg_hub, c))
            else: continue 

        for p in Products:
            surv = 1.0
            r = spoil_rate[p]
            for u, v in path_arcs:
                if u == "W" and v == "LGR": leg_r = r * 0.1 
                else: leg_r = r
                surv *= (1 - leg_r) ** eff_time[(u, v)]
            
            routes.append({"comm": c, "prod": p, "entry": entry, "surv": surv})

# =============================================================================
# 4) OPTIMIZATION MODEL (Dual Capacity Constraints)
# =============================================================================

model = Model("FCNQ_Real_Pallets")
model.Params.LogToConsole = 0

x = model.addVars(len(routes), lb=0, name="x")
missing = model.addVars([(c, p) for c in Communities for p in Products], lb=0, name="miss")
n_trucks_YUL = model.addVar(vtype=GRB.CONTINUOUS, lb=0, name="trucks_YUL")
n_trucks_LGR = model.addVar(vtype=GRB.CONTINUOUS, lb=0, name="trucks_LGR")

# Constraints
for c in Communities:
    for p in Products:
        delivered = quicksum(x[i] * routes[i]["surv"] for i in range(len(routes)) if routes[i]["comm"] == c and routes[i]["prod"] == p)
        model.addConstr(delivered + missing[c,p] >= demand_input[(c,p)])

model.addConstr(quicksum(x[i] for i in range(len(routes)) if routes[i]["entry"] == "YUL") <= hub_capacity["YUL"])
model.addConstr(quicksum(x[i] for i in range(len(routes)) if routes[i]["entry"] == "LGR") <= hub_capacity["LGR"])

# TRUCK CONSTRAINTS (Weight OR Volume)
model.addConstr(quicksum(x[i] for i in range(len(routes)) if routes[i]["entry"] == "YUL") <= n_trucks_YUL * TRUCK_MAX_WEIGHT)
model.addConstr(quicksum(x[i] for i in range(len(routes)) if routes[i]["entry"] == "LGR") <= n_trucks_LGR * TRUCK_MAX_WEIGHT)
model.addConstr(quicksum(x[i] / pallet_weights[routes[i]["prod"]] for i in range(len(routes)) if routes[i]["entry"] == "YUL") <= n_trucks_YUL * TRUCK_MAX_PALLETS)
model.addConstr(quicksum(x[i] / pallet_weights[routes[i]["prod"]] for i in range(len(routes)) if routes[i]["entry"] == "LGR") <= n_trucks_LGR * TRUCK_MAX_PALLETS)

# Objective
obj_ordered = quicksum(x[i] for i in range(len(routes)))
obj_missing = quicksum(missing[c,p] * 10000.0 for c in Communities for p in Products)
model.setObjective(obj_ordered + obj_missing, GRB.MINIMIZE)

model.optimize()

# =============================================================================
# 5) REPORTING (WITH TRUCK COUNT)
# =============================================================================

if model.status == GRB.OPTIMAL:
    val_ordered = obj_ordered.getValue()
    val_delivered = TOTAL_DEMAND - sum(missing[c,p].X for c in Communities for p in Products)
    
    print(f"\n========= FCNQ PALLET LOGISTICS REPORT =========")
    print(f"Total Demand:    {TOTAL_DEMAND:,.0f} lbs")
    print(f"Total Ordered:   {val_ordered:,.0f} lbs")
    print(f"Total Spoilage:  {val_ordered - val_delivered:,.0f} lbs")
    
    # --- METRIC 1: TRUCK FLEET REQUIREMENTS ---
    print("\n-------------------------------------------------------------")
    print("METRIC 1: TRUCK LOAD REQUIREMENTS (Weekly)")
    print("-------------------------------------------------------------")
    
    lgr_lbs = sum(x[i].X for i in range(len(routes)) if routes[i]["entry"] == "LGR")
    lgr_pallets = sum(x[i].X / pallet_weights[routes[i]["prod"]] for i in range(len(routes)) if routes[i]["entry"] == "LGR")
    
    yul_lbs = sum(x[i].X for i in range(len(routes)) if routes[i]["entry"] == "YUL")
    yul_pallets = sum(x[i].X / pallet_weights[routes[i]["prod"]] for i in range(len(routes)) if routes[i]["entry"] == "YUL")
    
    # Calculate exact trucks
    trucks_lgr_math = n_trucks_LGR.X
    trucks_yul_math = n_trucks_YUL.X
    
    # Round up for Physical Trucks
    physical_trucks_lgr = math.ceil(trucks_lgr_math)
    physical_trucks_yul = math.ceil(trucks_yul_math)
    
    print(f"LA GRANDE (LGR) ROUTE:")
    print(f"  - Total Cargo:   {lgr_lbs:,.0f} lbs")
    print(f"  - Total Volume:  {lgr_pallets:.1f} pallets")
    print(f"  - TRUCKS NEEDED: {physical_trucks_lgr} Trucks (Exact Load: {trucks_lgr_math:.2f})")
    
    # Identify bottleneck
    if trucks_lgr_math > 0:
        if (lgr_pallets / TRUCK_MAX_PALLETS) > (lgr_lbs / TRUCK_MAX_WEIGHT):
            print(f"  - Limiting Factor: VOLUME (Trucks 'cubed out' before weight limit)")
        else:
            print(f"  - Limiting Factor: WEIGHT (Trucks hit 44k lbs limit)")

    print(f"\nMONTREAL (YUL) ROUTE:")
    print(f"  - Total Cargo:   {yul_lbs:,.0f} lbs")
    print(f"  - Total Volume:  {yul_pallets:.1f} pallets")
    print(f"  - TRUCKS NEEDED: {physical_trucks_yul} Trucks (Exact Load: {trucks_yul_math:.2f})")

    # --- METRIC 2: SPOILAGE PER PRODUCT ---
    print("\n-------------------------------------------------------------")
    print("METRIC 2: SPOILAGE & ORDER BUFFER")
    print("-------------------------------------------------------------")
    print(f"{'Product':<10} | {'Demand (lbs)':<12} | {'Ordered':<10} | {'Buffer':<10}")
    for p in Products:
        p_dem = sum(demand_input.get((c, p), 0) for c in Communities)
        p_ord = sum(x[i].X for i in range(len(routes)) if routes[i]["prod"] == p)
        print(f"{p:<10} | {p_dem:<12.0f} | {p_ord:<10.0f} | {p_ord-p_dem:<10.0f}")
    
    # --- METRIC 3: TRUCK UTILIZATION EFFICIENCY ---
    print("\n-------------------------------------------------------------")
    print("METRIC 3: TRUCK UTILIZATION EFFICIENCY")
    print("-------------------------------------------------------------")
    # For LGR Trucks (The critical path)
    if trucks_lgr_math > 0:
        avg_weight = lgr_lbs / trucks_lgr_math
        avg_pallets = lgr_pallets / trucks_lgr_math
        
        weight_util = (avg_weight / TRUCK_MAX_WEIGHT) * 100
        vol_util = (avg_pallets / TRUCK_MAX_PALLETS) * 100
        
        print(f"LA GRANDE TRUCKS AVERAGE LOAD:")
        print(f"  - Weight Utilization: {weight_util:.1f}% ({avg_weight:,.0f} lbs / 44k lbs)")
        print(f"  - Volume Utilization: {vol_util:.1f}% ({avg_pallets:.1f} pallets / 26 spots)")
        
        if abs(weight_util - vol_util) > 20:
            print(f"  -> INSIGHT: Unbalanced Load. We are either 'Weighing Out' or 'Cubing Out' too early.")
        else:
            print(f"  -> INSIGHT: Perfect Mix. The truck is full in both weight and volume.")

    # --- METRIC 4: THE "COLD CHAIN BENEFIT" ---
    print("\n-------------------------------------------------------------")
    print("METRIC 4: COLD CHAIN SAVINGS (Why LGR?)")
    print("-------------------------------------------------------------")
    # Calculate how much spoilage was saved by using LGR Truck vs. Air
    # Concept: Compare spoilage rate of LGR path vs YUL path for total volume
    lgr_spoilage_rate = 0.0
    yul_spoilage_rate = 0.0
    
    # Calculate weighted avg spoilage rate for each route
    lgr_total_spoil = sum((x[i].X * (1 - routes[i]["surv"])) for i in range(len(routes)) if routes[i]["entry"] == "LGR")
    yul_total_spoil = sum((x[i].X * (1 - routes[i]["surv"])) for i in range(len(routes)) if routes[i]["entry"] == "YUL")
    
    if lgr_lbs > 0: lgr_spoilage_rate = (lgr_total_spoil / lgr_lbs) * 100
    if yul_lbs > 0: yul_spoilage_rate = (yul_total_spoil / yul_lbs) * 100
    
    print(f"  - Spoilage Rate via LGR (Truck+Air): {lgr_spoilage_rate:.2f}%")
    print(f"  - Spoilage Rate via YUL (Air Only):  {yul_spoilage_rate:.2f}%")
    
    if lgr_spoilage_rate < yul_spoilage_rate:
        saved_lbs = lgr_lbs * (yul_spoilage_rate - lgr_spoilage_rate) / 100
        print(f"  -> SUCCESS: The Refrigerated Truck saved ~{saved_lbs:.0f} lbs of food from rotting.")

    # --- METRIC 5: REGIONAL HUB BREAKDOWN ---
    print("\n-------------------------------------------------------------")
    print("METRIC 5: REGIONAL HUB DISTRIBUTION")
    print("-------------------------------------------------------------")
    hub_flows = {"LGR": 0.0, "Kuujjuaq": 0.0, "Puvirnituq": 0.0}
    for i in range(len(routes)):
        flow = x[i].X
        if flow < 0.1: continue
        entry = routes[i]["entry"]
        comm = routes[i]["comm"]
        
        # Determine Hub
        if entry == "LGR": h = "LGR"
        elif entry == "YUL":
            # If YUL, where did it fly?
            if comm in ["Akulivik", "Ivujivik", "Salluit", "Puvirnituq"]: h = "Puvirnituq"
            else: h = "Kuujjuaq"
        hub_flows[h] += flow

    print(f"{'Hub':<15} | {'Lbs Handled':<15} | {'% of Network'}")
    total_sys = sum(hub_flows.values())
    for h, vol in hub_flows.items():
        print(f"{h:<15} | {vol:<15,.0f} | {(vol/total_sys*100):.1f}%")

    # --- METRIC 6: THE BANANA vs TURNIP TEST ---
    print("\n-------------------------------------------------------------")
    print("METRIC 6: PRODUCT ROUTING CHECK (The 'Banana Test')")
    print("-------------------------------------------------------------")
    # Check split for Bananas (Fragile) vs Turnips ( hardy)
    for check_prod in ["Bananas", "Turnips", "Apples"]:
        p_lgr = sum(x[i].X for i in range(len(routes)) if routes[i]["prod"] == check_prod and routes[i]["entry"] == "LGR")
        p_yul = sum(x[i].X for i in range(len(routes)) if routes[i]["prod"] == check_prod and routes[i]["entry"] == "YUL")
        total = p_lgr + p_yul
        if total > 0:
            print(f"{check_prod:<10}: {p_lgr/total*100:.0f}% via LGR (Truck) | {p_yul/total*100:.0f}% via YUL (Jet)")

else:
    print("Optimization Failed.")

Set parameter LogToConsole to value 0

========= FCNQ PALLET LOGISTICS REPORT =========
Total Demand:    107,392 lbs
Total Ordered:   108,939 lbs
Total Spoilage:  1,547 lbs

-------------------------------------------------------------
METRIC 1: TRUCK LOAD REQUIREMENTS (Weekly)
-------------------------------------------------------------
LA GRANDE (LGR) ROUTE:
  - Total Cargo:   29,554 lbs
  - Total Volume:  10.4 pallets
  - TRUCKS NEEDED: 1 Trucks (Exact Load: 0.67)
  - Limiting Factor: WEIGHT (Trucks hit 44k lbs limit)

MONTREAL (YUL) ROUTE:
  - Total Cargo:   79,385 lbs
  - Total Volume:  28.0 pallets
  - TRUCKS NEEDED: 2 Trucks (Exact Load: 1.80)

-------------------------------------------------------------
METRIC 2: SPOILAGE & ORDER BUFFER
-------------------------------------------------------------
Product    | Demand (lbs) | Ordered    | Buffer    
Oranges    | 9965         | 10047      | 81        
Apples     | 35487        | 35776      | 289       
Bananas    | 29005       